# Прв ПИ Р1 5.4.2024

## Теодора Петровска, 98/2021

In [2]:
import sympy as sp
import script as rt

from IPython.display import display
from IPython.lib.display import IFrame

### Задача 1

#### Зглоб 4 треба да се изротира за 90 за да се добие ориентација на извршниот елемент како на цртежот. И зглоб 2 и зглоб 3 ако се изротираат за 90 ќе се добие како цртежот. Вредностите ги изразувам во метри наместо во центиметри, па така наместо 100 имам 1 и наместо 30 имам 0.3

In [6]:
theta1, theta2, theta3, d, d2, d3, d4, l1, l2, l3 = sp.symbols('theta1, theta2, theta3, d, d2, d3, d4, l1, l2, l3')
robot = rt.SerialLinkRobot()
robot.add_prismatic_joint(0, d, l1, sp.pi)
robot.add_revolute_joint(theta1, d2, l2, 0)
robot.add_revolute_joint(theta2, d3, l3, 0)
robot.add_revolute_joint(theta3, d4, 0, sp.pi/2)
robot.add_subs([(d2, 0.3), (d3, 0.3), (d4, 0.3), (l1, 1), (l2, 1), (l3, 1)])
robot.interact()

In [7]:
T = robot.get_dh_matrix()
T

Matrix([
[ cos(theta1 + theta2 + theta3),  0, sin(theta1 + theta2 + theta3), l1 + l2*cos(theta1) + l3*cos(theta1 + theta2)],
[-sin(theta1 + theta2 + theta3),  0, cos(theta1 + theta2 + theta3),     -l2*sin(theta1) - l3*sin(theta1 + theta2)],
[                             0, -1,                             0,                              d - d2 - d3 - d4],
[                             0,  0,                             0,                                             1]])

#### Работниот простор е цилиндар во којшто има цилиндри со помал радиус. Но, има ограничувања: транслаторниот зглоб не може да оди во минус, а ротационите зглобови не може да ротираат за 360 степени за да не удрат во раката.

### Задача 2

In [8]:
theta1, theta3, d3, l1 = sp.symbols('theta1, theta3, d3, l1')
robot = rt.SerialLinkRobot()
robot.add_revolute_joint(theta1, 0, 0, sp.pi/2)
robot.add_revolute_joint(theta2, 0, l1, 0)
robot.add_prismatic_joint(0, d3, 0, 0)
robot.add_subs([(l1, 1)])
robot.interact()

In [9]:
T = robot.get_dh_matrix()
T

Matrix([
[cos(theta1)*cos(theta2), -sin(theta2)*cos(theta1),  sin(theta1),  d3*sin(theta1) + l1*cos(theta1)*cos(theta2)],
[sin(theta1)*cos(theta2), -sin(theta1)*sin(theta2), -cos(theta1), -d3*cos(theta1) + l1*sin(theta1)*cos(theta2)],
[            sin(theta2),              cos(theta2),            0,                               l1*sin(theta2)],
[                      0,                        0,            0,                                            1]])

In [10]:
x, y, z = T[:3, 3]

In [15]:
def inverse_kinematics(xn, yn, zn):                     # на влез прима вредности за позицијата во којашто треба да застане извршниот елемент
    x, y, z = sp.symbols('x, y, z')                     # x, y и z ги дефинираме како симболи, долу истото го правиме за синусите и косинусите
    s1, c1, s2, c2 = sp.symbols('s1, c1, s2, c2')
    equations = [                                       # ги пишуваме равенките на транслација коишто ги земаме од DH моделот (тоа се првите 3 р-ки од последната колона)
        d3*s1 + l1*c1*c2 - x,
        -d3*c1 + l1*s1*c2 - y,
        l1*s2 - z,
        s1**2 + c1**2 - 1,
        s2**2 + c2**2 - 1]
    solutions = sp.nonlinsolve(equations, [s1, c1, s2, c2, d3])    # реши ги р-ките од погоре и изрази ги преку симболички променливи. Се решава за синуси и косинуси од аглите theta1 и theta2 и за d3
    display('Решенија изразени преку симболички променливи:')
    display(solutions)
    solutions_subbed = solutions.subs([(x, xn), (y, yn), (z, zn), (l1, 1)]) # замени ги р-ките што ги реши преку симболички променливи со вистински вредности -> за x, y и z земи ги вредностите што ги доби на влез, а за должина на краците задавам јас вредност 1
    for i, solution in enumerate(solutions_subbed):
        s1n, c1n, s2n, c2n, d3n = solution                         # земи ги вредностите за синусите и косинусите од аглите theta1 и theta2 и за d3
        theta1n = sp.deg(sp.atan2(s1n, c1n)).evalf(3)              # најди ја вредноста на theta1 преку пресметување на atan2 
        theta2n = sp.deg(sp.atan2(s2n, c2n)).evalf(3)              # најди ја вредноста на theta2 преку пресметување на atan2 
        d3n = d3n.evalf(3)                                         # најди ја вредноста на d3
        print(f'Решенија изразени преку нумерички вредности {i+1}: {(theta1n, theta2n, d3n)}')    # испечати ги на излез theta1, theta2, d3 редоследно


inverse_kinematics(xn=-0.5, yn=3, zn=-0.8)

'Решенија изразени преку симболички променливи:'

{((-x*sqrt(-l1**2 + x**2 + y**2 + z**2) - y*sqrt(l1**2 - z**2))/(x**2 + y**2), (-x*sqrt(l1**2 - z**2) + y*sqrt(-l1**2 + x**2 + y**2 + z**2))/(x**2 + y**2), z/l1, -sqrt(-(-l1 + z)*(l1 + z))/l1, -sqrt(-l1**2 + x**2 + y**2 + z**2)), ((-x*sqrt(-l1**2 + x**2 + y**2 + z**2) + y*sqrt(l1**2 - z**2))/(x**2 + y**2), (x*sqrt(l1**2 - z**2) + y*sqrt(-l1**2 + x**2 + y**2 + z**2))/(x**2 + y**2), z/l1, sqrt(-(-l1 + z)*(l1 + z))/l1, -sqrt(-l1**2 + x**2 + y**2 + z**2)), ((x*sqrt(-l1**2 + x**2 + y**2 + z**2) - y*sqrt(l1**2 - z**2))/(x**2 + y**2), (-x*sqrt(l1**2 - z**2) - y*sqrt(-l1**2 + x**2 + y**2 + z**2))/(x**2 + y**2), z/l1, -sqrt(-(-l1 + z)*(l1 + z))/l1, sqrt(-l1**2 + x**2 + y**2 + z**2)), ((x*sqrt(-l1**2 + x**2 + y**2 + z**2) + y*sqrt(l1**2 - z**2))/(x**2 + y**2), (x*sqrt(l1**2 - z**2) - y*sqrt(-l1**2 + x**2 + y**2 + z**2))/(x**2 + y**2), z/l1, sqrt(-(-l1 + z)*(l1 + z))/l1, sqrt(-l1**2 + x**2 + y**2 + z**2))}

Решенија изразени преку нумерички вредности 1: (-159., -127., 2.98)
Решенија изразени преку нумерички вредности 2: (-1.92, -127., -2.98)
Решенија изразени преку нумерички вредности 3: (178., -53.1, 2.98)
Решенија изразени преку нумерички вредности 4: (20.8, -53.1, -2.98)


#### Овој пристап дава и неточни решенија, тоа е поради тоа што функцијата не знае дека нашиот призматичен зглоб не може да прима негативни решенија, па ги зема и тие резултати предвид. Така, доколку ја земеме точката (-0.5, 3, -0.8) и ги погледнеме решенијата што нашата функција ни ги дава, ќе воочиме дека само првото и третото решение се точни бидејќи само кај нив призматичниот зглоб има позитивна вредност (третата вредност).

### Задача 4

#### Остатокот е напишан во физичката тетратка, ова е само за да се најде позицијата на точката

In [23]:
Tb = sp.Matrix([
    [0.82, 0, 0.57, 0],
    [0, 1, 0, 0],
    [-0.57, 0, 0.82, 0],
    [0, 0, 0, 1],
])
Tb

Matrix([
[ 0.82, 0, 0.57, 0],
[    0, 1,    0, 0],
[-0.57, 0, 0.82, 0],
[    0, 0,    0, 1]])

In [24]:
point = rt.hpoint3(0, 0, 5)
r = Tb * point
r.evalf()

Matrix([
[2.85],
[   0],
[ 4.1],
[ 1.0]])